# DATA 266 — Lab 2 — Part 2: fine-tuning Stable Diffusion 1.5 (Shriram Dundigalla)

## Cell 1 — run information (required by C1.4)

In [ ]:
# Cell 1 - run information (required)
import subprocess, sys, random, numpy as np, torch

SEED = 266
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip() or "n/a (not a git checkout)"
print({"commit": commit,
       "python": sys.version.split()[0],
       "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
       "seed": SEED})

## Install

Colab already ships torch; everything else is pinned here so the run is reproducible.
Restart the session if Colab asks you to, then carry on from the next cell.

In [ ]:
%pip install -q \
    diffusers==0.31.0 \
    transformers==4.46.3 \
    accelerate==1.1.1 \
    peft==0.13.2 \
    datasets==3.1.0 \
    huggingface_hub==0.26.2 \
    safetensors==0.4.5 \
    torchmetrics==1.5.2 \
    torch-fidelity==0.3.0 \
    lpips==0.1.4 \
    ImageHash==4.3.1
print("installed")

## Settings and paths

Copy the `Lab2` folder of the repository into your Drive, then point `PROJECT_DIR` at it.
Everything this notebook produces is written under `PROJECT_DIR/outputs/part2_shriram/`, so a
Colab disconnect does not lose the results.

In [ ]:
import json, time, io, hashlib, os, re, shutil
from pathlib import Path

import pandas as pd
from PIL import Image

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/data266_lab2")
    SEARCH_ROOTS = [PROJECT_DIR, Path("/content/drive/MyDrive")]
else:
    PROJECT_DIR = Path("/content/data266_lab2")
    SEARCH_ROOTS = [PROJECT_DIR, Path("/content")]


def find(name, roots, depth=4):
    """The folder holding `name`, searched breadth-first from each root in turn.

    Uploading into Drive by hand rarely reproduces a folder tree exactly, and a wrong path here
    wastes a whole session. Nearest root wins, so a copy inside the project beats a stray one.
    """
    for root in roots:
        if not root.exists():
            continue
        for level in range(depth + 1):
            for hit in sorted(root.glob("/".join(["*"] * level + [name]))):
                return hit.parent
    return None


DATA_DIR = find("prompts.json", SEARCH_ROOTS)
assert DATA_DIR, f"prompts.json was not found under any of {[str(r) for r in SEARCH_ROOTS]}"
print("prompts :", DATA_DIR / "prompts.json")
OUT_DIR = PROJECT_DIR / "outputs" / "part2_shriram"
for sub in ["generated", "grids", "metrics", "safety", "human_eval/ratings", "runs"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

# --- experiment settings (every number below is reported, nothing is tuned on the test set) ---
DATASET_NAME = "reach-vb/pokemon-blip-captions"   # cc-by-nc-sa-4.0, 833 captioned images
BASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
CLIP_NAME = "openai/clip-vit-base-patch16"
RESOLUTION = 512
TEST_FRACTION = 0.10
GEN_SEEDS = [0, 1]          # two images per prompt
GEN_STEPS = 30
GUIDANCE = 7.5

# Two runs on the same data and the same step budget, differing only in LoRA rank. Rank is the
# number of columns in the low-rank update, so it is the capacity knob: r=8 is the light run and
# r=32 has four times the adapter parameters to fit the same 3000 steps.
LORA_RANKS = [8, 32]
LORA_STEPS = 3000
LORA_LR = 1e-4
LORA_BATCH = 1
LORA_GRAD_ACCUM = 4

HF_USER = "CHANGE_ME"       # your Hugging Face username
# Paste the write token here while running, then clear it again before committing. Secrets in a
# committed notebook are an explicit DON'T in the rubric, and anyone who reads the repository
# gets full write access to the Hugging Face account.
HF_TOKEN = ""               # <-- paste here at run time, leave empty in the saved file
HF_PRIVATE = False          # the model card and dataset links go in links.txt, so keep them public
def hf_model_repo(rank):
    return f"{HF_USER}/pokemon-sd15-lora-r{rank}"
HF_IMAGE_REPO = f"{HF_USER}/data266-lab2-part2-images"

print("project :", PROJECT_DIR)
print("outputs :", OUT_DIR)
print("20 fixed prompts, written and committed before training began")

## 2.1 Data preparation

The dataset is **Pokémon BLIP captions** (`reach-vb/pokemon-blip-captions`), 833 images of
Pokémon artwork with captions written by a BLIP model. License `cc-by-nc-sa-4.0`, and the
characters themselves are copyrighted, so everything here is non-commercial research only.

Cleaning does two things and reports both counts:

1. **Corrupted images** — every image is fully decoded and its caption checked for being
   non-empty. Anything that fails is dropped.
2. **Duplicates** — exact duplicates are caught by an MD5 of the resized pixels, and
   near-duplicates by a 16-bit average hash (`ImageHash`) with a Hamming distance of 0.

In [ ]:
from datasets import load_dataset
import imagehash

raw = load_dataset(DATASET_NAME, split="train")
print("rows in the raw dataset:", len(raw), "| columns:", raw.column_names)


def to_rgb(image):
    """RGB copy of an image, with any transparency flattened onto white."""
    if image.mode in ("RGBA", "LA", "P"):
        image = image.convert("RGBA")
        background = Image.new("RGBA", image.size, (255, 255, 255, 255))
        image = Image.alpha_composite(background, image)
    return image.convert("RGB")


records, corrupted = [], []
for i in range(len(raw)):
    row = raw[i]
    caption = (row.get("text") or "").strip()
    try:
        image = to_rgb(row["image"])
        image.load()
        if min(image.size) < 64 or not caption:
            raise ValueError("too small or empty caption")
        resized = image.resize((RESOLUTION, RESOLUTION), Image.Resampling.LANCZOS)
    except Exception as exc:
        corrupted.append({"index": i, "reason": f"{type(exc).__name__}: {exc}"})
        continue
    records.append({"index": i,
                    "caption": caption,
                    "image": resized,
                    "md5": hashlib.md5(resized.tobytes()).hexdigest(),
                    "ahash": imagehash.average_hash(resized, hash_size=16)})

print("corrupted or unusable removed:", len(corrupted))
print("remaining:", len(records))

In [ ]:
# Duplicates: exact pixel match first, then perceptual near-duplicates.
seen_md5, kept, duplicates = {}, [], []
for record in records:
    if record["md5"] in seen_md5:
        duplicates.append({"index": record["index"], "kind": "exact",
                           "duplicate_of": seen_md5[record["md5"]]})
        continue
    seen_md5[record["md5"]] = record["index"]
    kept.append(record)

final, near_duplicates = [], []
for record in kept:
    match = next((k for k in final if record["ahash"] - k["ahash"] == 0), None)
    if match is not None:
        near_duplicates.append({"index": record["index"], "kind": "perceptual",
                                "duplicate_of": match["index"]})
        continue
    final.append(record)

duplicates += near_duplicates
cleaning_summary = {
    "dataset": DATASET_NAME,
    "rows_raw": len(raw),
    "corrupted_removed": len(corrupted),
    "exact_duplicates_removed": sum(d["kind"] == "exact" for d in duplicates),
    "perceptual_duplicates_removed": sum(d["kind"] == "perceptual" for d in duplicates),
    "rows_clean": len(final),
}
(OUT_DIR / "metrics" / "cleaning_summary.json").write_text(json.dumps(cleaning_summary, indent=2))
print(json.dumps(cleaning_summary, indent=2))

In [ ]:
# 90 / 10 split at seed 266. The test images are only ever used for evaluation.
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(range(len(final)), test_size=TEST_FRACTION,
                                       random_state=SEED, shuffle=True)
train_records = [final[i] for i in sorted(train_idx)]
test_records = [final[i] for i in sorted(test_idx)]

for name, rows in [("train", train_records), ("test", test_records)]:
    pd.DataFrame([{"source_index": r["index"], "caption": r["caption"]} for r in rows]).to_csv(
        OUT_DIR / "metrics" / f"split_{name}.csv", index=False)

print(f"train: {len(train_records)}   test: {len(test_records)}   resolution: {RESOLUTION}x{RESOLUTION}")

# 2.3.1 asks for in-domain prompts drawn from test-set captions. Ours were written from the
# dataset's caption style before the split existed, so the overlap is measured rather than
# asserted: each one is matched to its nearest test caption by word overlap.
STOP = {"a", "an", "the", "of", "with", "on", "in", "and", "is", "it", "its", "that", "to"}


def words(text):
    return {w for w in re.findall(r"[a-z]+", text.lower()) if w not in STOP}


test_captions = [r["caption"] for r in test_records]
rows = []
for item in json.loads((DATA_DIR / "prompts.json").read_text()):
    if item["type"] != "in_domain":
        continue
    best = max(test_captions, key=lambda c: len(words(c) & words(item["prompt"])) /
               max(len(words(c) | words(item["prompt"])), 1))
    overlap = len(words(best) & words(item["prompt"])) / max(len(words(best) | words(item["prompt"])), 1)
    rows.append({"id": item["id"], "prompt": item["prompt"],
                 "nearest_test_caption": best, "word_overlap": round(overlap, 2)})

prompt_overlap = pd.DataFrame(rows)
prompt_overlap.to_csv(OUT_DIR / "metrics" / "in_domain_prompt_overlap.csv", index=False)
print("median word overlap with the nearest test caption:",
      round(prompt_overlap["word_overlap"].median(), 2))
display(prompt_overlap)

In [ ]:
# A look at the data, and the two numbers behind the limitations written below.
import textwrap
import matplotlib.pyplot as plt
from collections import Counter

lengths = [len(r["caption"].split()) for r in final]
words = Counter(w for r in final for w in r["caption"].lower().split())
print(f"caption length in words: mean {np.mean(lengths):.1f}, median {np.median(lengths):.0f}, "
      f"min {min(lengths)}, max {max(lengths)}")
print("ten most common caption words:", words.most_common(10))

sample = random.Random(SEED).sample(train_records, 8)
figure, axes = plt.subplots(2, 4, figsize=(14, 8))
for axis, record in zip(axes.ravel(), sample):
    axis.imshow(record["image"]); axis.axis("off")
    axis.set_title(textwrap.fill(record["caption"], 32), fontsize=8)
figure.suptitle("Pokemon BLIP captions — eight training images after cleaning")
figure.tight_layout(); plt.show()

### Dataset description, and two limitations

833 single-subject illustrations of Pokémon characters, almost all a creature centred on a plain
light background, each with one short BLIP-generated caption. After cleaning the set is split
747 / 83 at seed 266 and every image is resized to 512×512, the native resolution of SD 1.5.

1. **The captions are machine-written and often wrong.** BLIP describes what it guesses, so
   colours and species are frequently mislabelled ("a cartoon bird" for a fish) and the vocabulary
   is tiny — the counts above show how few distinct words carry most of the captions. A model
   trained on them learns the *look* of the data far better than it learns to follow a prompt, so
   a CLIP score drop after fine-tuning is expected rather than surprising.
2. **The imagery is narrow and the characters are copyrighted.** One subject, one pose, one plain
   background, and no humans, scenes or text. Anything the prompt asks for outside that envelope
   (counting, spatial arrangement, photographic scenes) is outside the distribution entirely. The
   `cc-by-nc-sa-4.0` license and the copyright on the characters restrict every model and image
   here to non-commercial research.

## 2.2 Fine-tuning — two LoRA ranks on the same data and step budget

Two runs, identical except for the LoRA rank: `r=8` and `r=32`, both for 3000 steps on the same
90% training split. Rank sets how many columns the low-rank update has, so it is the capacity
knob — `r=32` has four times the adapter parameters to fit in the same step budget, which is the
trade-off this section measures.

Both use the unmodified `diffusers` example script. Finished adapters are written to Drive, so if
the session drops, rerunning this notebook picks up any run that already completed instead of
retraining it. Expect roughly an hour per run.


In [ ]:
train_dir = Path("/content/pokemon_train")

# The diffusers script reads an imagefolder with a metadata.jsonl of {file_name, text}.
train_dir.mkdir(parents=True, exist_ok=True)
with open(train_dir / "metadata.jsonl", "w") as handle:
    for i, record in enumerate(train_records):
        name = f"{i:05d}.png"
        record["image"].save(train_dir / name)
        handle.write(json.dumps({"file_name": name, "text": record["caption"]}) + "\n")
print("wrote", len(train_records), "training images to", train_dir)

In [ ]:
import subprocess, threading
from safetensors.torch import load_file

SCRIPT = Path("/content/train_text_to_image_lora.py")
if not SCRIPT.exists():
    !wget -q -O {SCRIPT} https://raw.githubusercontent.com/huggingface/diffusers/v0.31.0/examples/text_to_image/train_text_to_image_lora.py


def watch_gpu(stop, readings):
    """Peak memory of the training process. accelerate launches a child process, so
    torch.cuda.max_memory_allocated() in this notebook would report zero."""
    while not stop.wait(5):
        probe = subprocess.run(["nvidia-smi", "--query-gpu=memory.used",
                                "--format=csv,noheader,nounits"], capture_output=True, text=True)
        line = probe.stdout.strip().splitlines()
        if line:
            readings.append(int(line[0]))


def train_one(rank):
    """One LoRA run. Finished adapters live in Drive, so a dropped session resumes here."""
    out_dir = OUT_DIR / "runs" / f"lora_r{rank}_{LORA_STEPS}steps"
    weights = out_dir / "pytorch_lora_weights.safetensors"
    log_path = OUT_DIR / "runs" / f"lora_r{rank}_train.log"

    if weights.exists():
        print(f"rank {rank}: adapter already in Drive, skipping the run")
        record = json.loads((out_dir / "run_record.json").read_text())
    else:
        out_dir.mkdir(parents=True, exist_ok=True)
        command = [
            "accelerate", "launch", "--mixed_precision=fp16", str(SCRIPT),
            f"--pretrained_model_name_or_path={BASE_MODEL}",
            f"--train_data_dir={train_dir}", "--caption_column=text",
            f"--resolution={RESOLUTION}", "--random_flip",
            f"--train_batch_size={LORA_BATCH}",
            f"--gradient_accumulation_steps={LORA_GRAD_ACCUM}",
            f"--max_train_steps={LORA_STEPS}",
            f"--learning_rate={LORA_LR}", "--lr_scheduler=cosine", "--lr_warmup_steps=0",
            f"--rank={rank}", f"--seed={SEED}", "--dataloader_num_workers=2",
            f"--output_dir={out_dir}",
        ]
        stop, readings = threading.Event(), []
        watcher = threading.Thread(target=watch_gpu, args=(stop, readings), daemon=True)
        watcher.start()
        started = time.time()
        with open(log_path, "w") as log:
            finished = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, text=True)
        seconds = time.time() - started
        stop.set(); watcher.join()

        if finished.returncode != 0:
            print(log_path.read_text()[-2500:])
            raise RuntimeError(f"rank {rank} training failed; the log tail is above")

        record = {"run": f"lora_r{rank}", "rank": rank,
                  "learning_rate": LORA_LR, "steps": LORA_STEPS,
                  "effective_batch_size": LORA_BATCH * LORA_GRAD_ACCUM,
                  "peak_gpu_memory_gb": round(max(readings, default=0) / 1024, 2),
                  "training_seconds": round(seconds, 1),
                  "gpu_hours": round(seconds / 3600, 3)}
        (out_dir / "run_record.json").write_text(json.dumps(record, indent=2))

    # counted from the saved adapter, so the number cannot drift from the weights that were pushed
    record["trainable_parameters"] = sum(t.numel() for t in load_file(weights).values())
    record["path"] = str(out_dir)
    return record


training_table = pd.DataFrame([train_one(rank) for rank in LORA_RANKS])
training_table = training_table[["run", "rank", "trainable_parameters", "learning_rate", "steps",
                                 "effective_batch_size", "peak_gpu_memory_gb",
                                 "training_seconds", "gpu_hours"]]
training_table.to_csv(OUT_DIR / "metrics" / "training_table.csv", index=False)
display(training_table)

### Push both adapters to Hugging Face (2.2.3)

In [ ]:
import getpass
from huggingface_hub import HfApi, login

assert HF_USER != "CHANGE_ME", "set HF_USER in the settings cell first"
# Colab secret first, then the settings cell, then a masked prompt. Nothing is ever printed.
token = os.environ.get("HF_TOKEN") or HF_TOKEN
try:
    from google.colab import userdata
    token = token or userdata.get("HF_TOKEN")
except Exception:
    pass
login(token=token or getpass.getpass("Hugging Face write token: "))
api = HfApi()

for rank in LORA_RANKS:
    repo = hf_model_repo(rank)
    api.create_repo(repo, repo_type="model", private=HF_PRIVATE, exist_ok=True)
    api.upload_folder(folder_path=str(OUT_DIR / "runs" / f"lora_r{rank}_{LORA_STEPS}steps"),
                      repo_id=repo, repo_type="model",
                      allow_patterns=["*.safetensors", "*.json"])
    print("pushed", repo)

## 2.3 Image generation

The 20 prompts in `data/part2/prompts.json` were fixed and committed before any training —
10 in-domain (taken from test-set captions), 5 out-of-domain and 5 compositional. Every model
sees the same prompts, the same two seeds and the same sampler settings, so 40 images per model.

In [ ]:
from diffusers import StableDiffusionPipeline

prompts = json.loads((DATA_DIR / "prompts.json").read_text())
print(len(prompts), "prompts:", Counter(p["type"] for p in prompts))

pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, safety_checker=None, requires_safety_checker=False
).to("cuda")
pipe.set_progress_bar_config(disable=True)

MODELS = {"base": None}
for rank in LORA_RANKS:
    MODELS[f"lora_r{rank}"] = str(OUT_DIR / "runs" / f"lora_r{rank}_{LORA_STEPS}steps")
print("models to generate with:", list(MODELS))

In [ ]:
generation_log, images, lora_loaded = [], {}, False

for model_name, weights in MODELS.items():
    if weights is None and lora_loaded:
        pipe.unload_lora_weights()
        lora_loaded = False
    elif weights is not None:
        pipe.load_lora_weights(weights, weight_name="pytorch_lora_weights.safetensors")
        lora_loaded = True
    folder = OUT_DIR / "generated" / model_name
    folder.mkdir(parents=True, exist_ok=True)

    for prompt in prompts:
        for seed in GEN_SEEDS:
            generator = torch.Generator("cuda").manual_seed(seed)
            started = time.time()
            image = pipe(prompt["prompt"], num_inference_steps=GEN_STEPS,
                         guidance_scale=GUIDANCE, generator=generator).images[0]
            seconds = time.time() - started
            path = folder / f"{prompt['id']}_seed{seed}.png"
            image.save(path)
            images[(model_name, prompt["id"], seed)] = image
            generation_log.append({"model": model_name, "prompt_id": prompt["id"],
                                   "prompt_type": prompt["type"], "prompt": prompt["prompt"],
                                   "seed": seed, "steps": GEN_STEPS, "guidance": GUIDANCE,
                                   "seconds": round(seconds, 2), "path": str(path)})
    print(f"{model_name}: {len(prompts) * len(GEN_SEEDS)} images")

generation_log = pd.DataFrame(generation_log)
generation_log.to_csv(OUT_DIR / "metrics" / "generation_log.csv", index=False)
print("median seconds per image:", round(generation_log["seconds"].median(), 2))

In [ ]:
# Side-by-side grid for 8 prompts (3 in-domain, 3 out-of-domain, 2 compositional), seed 0.
GRID_PROMPTS = ["P01", "P04", "P10", "P11", "P13", "P15", "P16", "P19"]
prompt_text = {p["id"]: p["prompt"] for p in prompts}
model_names = list(MODELS)

figure, axes = plt.subplots(len(GRID_PROMPTS), len(model_names),
                            figsize=(3.2 * len(model_names), 3.4 * len(GRID_PROMPTS)),
                            squeeze=False)
for row, prompt_id in enumerate(GRID_PROMPTS):
    for column, model_name in enumerate(model_names):
        axis = axes[row][column]
        axis.imshow(images[(model_name, prompt_id, 0)]); axis.axis("off")
        if row == 0:
            axis.set_title(model_name, fontsize=11)
    axes[row][0].text(-0.05, 0.5, f"{prompt_id}\n{prompt_text[prompt_id][:40]}...",
                      transform=axes[row][0].transAxes, ha="right", va="center", fontsize=8)
figure.suptitle("Same prompt, same seed, one column per model", y=1.001)
figure.tight_layout()
figure.savefig(OUT_DIR / "grids" / "side_by_side.png", dpi=110, bbox_inches="tight")
plt.show()

## 2.4 Evaluation

| Measurement | How it is computed here |
|---|---|
| Primary | **KID** between the 20 in-domain generations and the 83 test images, and **CLIP score** (100 × cosine similarity) between each prompt and its image |
| Secondary | **LPIPS diversity**: the distance between the two seeds of each prompt |
| Human evaluation | blinded 25-pair comparison, prepared below and scored by three raters |
| Efficiency | seconds per image, and GPU-hours if the optional training run was used |

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained(CLIP_NAME).to("cuda").eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_NAME)


def clip_scores(image_list, text_list, batch=16):
    """100 x cosine similarity between each image and its own text."""
    scores = []
    for start in range(0, len(image_list), batch):
        inputs = clip_processor(text=text_list[start:start + batch],
                                images=image_list[start:start + batch],
                                return_tensors="pt", padding=True,
                                truncation=True, max_length=77).to("cuda")
        with torch.no_grad():
            output = clip_model(**inputs)
        scores += torch.diagonal(output.logits_per_image).tolist()
    return scores


generation_log["clip_score"] = clip_scores(
    [images[(r.model, r.prompt_id, r.seed)] for r in generation_log.itertuples()],
    generation_log["prompt"].tolist())
print(generation_log.groupby(["model", "prompt_type"])["clip_score"].mean().unstack().round(2))

In [ ]:
from torchmetrics.image.kid import KernelInceptionDistance


def as_uint8(image_list):
    array = np.stack([np.asarray(im.resize((299, 299), Image.Resampling.BILINEAR)) for im in image_list])
    return torch.from_numpy(array).permute(0, 3, 1, 2).to(torch.uint8)


real_batch = as_uint8([r["image"] for r in test_records])
kid_results = {}
for model_name in MODELS:
    rows = generation_log[(generation_log["model"] == model_name) &
                          (generation_log["prompt_type"] == "in_domain")]
    fake_batch = as_uint8([images[(r.model, r.prompt_id, r.seed)] for r in rows.itertuples()])
    kid = KernelInceptionDistance(subset_size=20, subsets=100, reset_real_features=False, normalize=False)
    kid.update(real_batch, real=True)
    kid.update(fake_batch, real=False)
    mean, std = kid.compute()
    kid_results[model_name] = (float(mean), float(std))
    print(f"{model_name}: KID {float(mean):.4f} +/- {float(std):.4f}  "
          f"({len(fake_batch)} in-domain generations vs {len(real_batch)} test images)")
del real_batch
torch.cuda.empty_cache()

In [ ]:
import lpips

lpips_model = lpips.LPIPS(net="alex", verbose=False).to("cuda")


def lpips_tensor(image):
    array = np.asarray(image.resize((256, 256), Image.Resampling.BILINEAR), dtype=np.float32) / 127.5 - 1.0
    return torch.from_numpy(array).permute(2, 0, 1)[None].to("cuda")


diversity = []
for model_name in MODELS:
    for prompt in prompts:
        a, b = (images[(model_name, prompt["id"], s)] for s in GEN_SEEDS)
        with torch.no_grad():
            distance = float(lpips_model(lpips_tensor(a), lpips_tensor(b)))
        diversity.append({"model": model_name, "prompt_id": prompt["id"],
                          "prompt_type": prompt["type"], "lpips": distance})
diversity = pd.DataFrame(diversity)
diversity.to_csv(OUT_DIR / "metrics" / "diversity_per_prompt.csv", index=False)
print(diversity.groupby("model")["lpips"].mean().round(4))

In [ ]:
# One results table for Part 2.
results = pd.DataFrame([{
    "model": model_name,
    "role": "baseline" if model_name == "base" else f"fine-tuned, {LORA_STEPS} steps",
    "kid_in_domain": round(kid_results[model_name][0], 4),
    "kid_std": round(kid_results[model_name][1], 4),
    "clip_score": round(generation_log[generation_log["model"] == model_name]["clip_score"].mean(), 2),
    "clip_in_domain": round(generation_log[(generation_log["model"] == model_name) &
                                           (generation_log["prompt_type"] == "in_domain")]["clip_score"].mean(), 2),
    "lpips_diversity": round(diversity[diversity["model"] == model_name]["lpips"].mean(), 4),
    "median_seconds_per_image": round(generation_log[generation_log["model"] == model_name]["seconds"].median(), 2),
} for model_name in MODELS])

results = results.merge(training_table[["run", "trainable_parameters", "gpu_hours",
                                        "peak_gpu_memory_gb"]],
                        left_on="model", right_on="run", how="left").drop(columns="run")
results.to_csv(OUT_DIR / "metrics" / "results_table.csv", index=False)
display(results)

# KID is the primary quality metric, so the run with the lowest KID is the one carried into the
# blinded comparison, the model card and Part 3. Chosen here rather than by eye.
runs_only = results[results["model"] != "base"]
FINAL_RUN = runs_only.loc[runs_only["kid_in_domain"].idxmin(), "model"]
print("final run selected on in-domain KID:", FINAL_RUN)

### Failure modes

Both examples are chosen by the numbers rather than by eye: the prompt with the lowest CLIP score,
and the prompt whose two seeds are closest in LPIPS (the least diverse pair).

In [ ]:
per_prompt = generation_log.groupby(["model", "prompt_id"])["clip_score"].mean()
worst_prompt = per_prompt["base"].idxmin()
least_diverse = diversity[diversity["model"] == "base"].set_index("prompt_id")["lpips"].idxmin()

figure, axes = plt.subplots(2, 2, figsize=(8, 8.8))
for row, (prompt_id, label) in enumerate([(worst_prompt, f"lowest CLIP ({per_prompt['base'][worst_prompt]:.1f})"),
                                          (least_diverse, "least diverse pair")]):
    for column, seed in enumerate(GEN_SEEDS):
        axes[row][column].imshow(images[("base", prompt_id, seed)]); axes[row][column].axis("off")
        axes[row][column].set_title(f"{prompt_id} seed {seed}", fontsize=9)
    axes[row][0].text(-0.04, 0.5, f"{label}\n{prompt_text[prompt_id][:45]}...",
                      transform=axes[row][0].transAxes, ha="right", va="center", fontsize=8)
figure.suptitle("Base SD 1.5 — two failure modes")
figure.tight_layout()
figure.savefig(OUT_DIR / "grids" / "failure_modes.png", dpi=110, bbox_inches="tight")
plt.show()

print("worst prompt :", worst_prompt, "|", prompt_text[worst_prompt])
print("least diverse:", least_diverse, "|", prompt_text[least_diverse])

### Safety (C4): license, NSFW filter, memorization

**License.** `cc-by-nc-sa-4.0` on the dataset and copyrighted characters in the images, so this
is non-commercial research and education only. SD 1.5 itself is CreativeML OpenRAIL-M.

**NSFW filter.** Every generated image goes through the Stable Diffusion safety checker. The
flags are recorded rather than acted on blindly, because the checker is known to fire on harmless
cartoon creatures.

**Memorization check.** For 10 in-domain generations the nearest *training* image is found by
CLIP cosine similarity, and the three closest pairs are shown. The comparison that matters is
against the nearest-neighbour similarity of unseen *test* images: if the generations are no closer
to the training set than real held-out images are, there is no sign of memorization.

In [ ]:
from diffusers.pipelines.stable_diffusion.safety_checker import StableDiffusionSafetyChecker
from transformers import CLIPImageProcessor

safety_checker = StableDiffusionSafetyChecker.from_pretrained(
    "CompVis/stable-diffusion-safety-checker", torch_dtype=torch.float16).to("cuda").eval()
safety_extractor = CLIPImageProcessor.from_pretrained("CompVis/stable-diffusion-safety-checker")

nsfw_rows = []
for key, image in images.items():
    pixels = safety_extractor([image], return_tensors="pt").pixel_values.to("cuda", torch.float16)
    array = np.asarray(image, dtype=np.float32)[None] / 255.0
    with torch.no_grad():
        _, flagged = safety_checker(clip_input=pixels, images=array)
    nsfw_rows.append({"model": key[0], "prompt_id": key[1], "seed": key[2], "flagged": bool(flagged[0])})

nsfw = pd.DataFrame(nsfw_rows)
nsfw.to_csv(OUT_DIR / "safety" / "nsfw_check.csv", index=False)
print(f"flagged by the safety checker: {int(nsfw['flagged'].sum())} of {len(nsfw)}")
display(nsfw[nsfw["flagged"]])

In [ ]:
@torch.no_grad()
def clip_embeddings(image_list, batch=32):
    out = []
    for start in range(0, len(image_list), batch):
        inputs = clip_processor(images=image_list[start:start + batch], return_tensors="pt").to("cuda")
        features = clip_model.get_image_features(**inputs)
        out.append(torch.nn.functional.normalize(features, dim=-1).cpu())
    return torch.cat(out)


train_embeddings = clip_embeddings([r["image"] for r in train_records])
in_domain_ids = [p["id"] for p in prompts if p["type"] == "in_domain"]
generated = [images[("base", pid, 0)] for pid in in_domain_ids]
similarity = clip_embeddings(generated) @ train_embeddings.T

memorization = pd.DataFrame({
    "prompt_id": in_domain_ids,
    "nearest_train_index": [train_records[int(i)]["index"] for i in similarity.argmax(1)],
    "clip_similarity": similarity.max(1).values.round(4).tolist(),
})
# The control: how close an unseen test image is to its nearest training image.
test_similarity = (clip_embeddings([r["image"] for r in test_records]) @ train_embeddings.T).max(1).values
memorization.to_csv(OUT_DIR / "safety" / "memorization_check.csv", index=False)

print(f"generated vs nearest training image : mean {memorization['clip_similarity'].mean():.4f}, "
      f"max {memorization['clip_similarity'].max():.4f}")
print(f"held-out test image vs nearest train: mean {float(test_similarity.mean()):.4f}, "
      f"max {float(test_similarity.max()):.4f}")
print("No memorization signal if the generated row is not above the test row.")

top3 = memorization.nlargest(3, "clip_similarity")
figure, axes = plt.subplots(3, 2, figsize=(6.5, 10))
for row, item in enumerate(top3.itertuples()):
    axes[row][0].imshow(images[("base", item.prompt_id, 0)])
    axes[row][0].set_title(f"generated {item.prompt_id}", fontsize=9)
    nearest = next(r for r in train_records if r["index"] == item.nearest_train_index)
    axes[row][1].imshow(nearest["image"])
    axes[row][1].set_title(f"nearest training image (CLIP {item.clip_similarity:.3f})", fontsize=9)
    for axis in axes[row]:
        axis.axis("off")
figure.suptitle("Three closest generated / training pairs")
figure.tight_layout()
figure.savefig(OUT_DIR / "safety" / "memorization_pairs.png", dpi=110, bbox_inches="tight")
plt.show()

### Human evaluation sheet

A blinded comparison needs two models, so this cell builds the rating sheet only when a second
run exists. It picks 25 prompt-and-seed pairs, randomizes which model is on the left, and writes
an answer key plus an empty sheet per rater. Three raters fill them in (both team members plus one
person outside the team) and the next cell computes the win rate from whatever sheets are present.

In [ ]:
RATERS = ["Shriram", "Tejas", "Outside"]
comparison_run = FINAL_RUN      # the run chosen on KID, compared blind against the base model

if comparison_run is None:
    print("Only the base model was generated, so there is no pair to compare.\n"
          "The graded blinded comparison for Part 2 is the one in Tejas's notebook "
          "(final LoRA run vs base model); my ratings for it are in the team repository.")
else:
    rng = random.Random(SEED)
    pairs = rng.sample([(p["id"], s) for p in prompts for s in GEN_SEEDS], 25)
    key_rows, sheet_rows = [], []
    for n, (prompt_id, seed) in enumerate(pairs, start=1):
        left_is_run = rng.random() < 0.5
        pair_dir = OUT_DIR / "human_eval" / f"pair_{n:02d}"
        pair_dir.mkdir(parents=True, exist_ok=True)
        left_model = comparison_run if left_is_run else "base"
        right_model = "base" if left_is_run else comparison_run
        images[(left_model, prompt_id, seed)].save(pair_dir / "left.png")
        images[(right_model, prompt_id, seed)].save(pair_dir / "right.png")
        key_rows.append({"pair": n, "prompt_id": prompt_id, "seed": seed,
                         "left_model": left_model, "right_model": right_model})
        sheet_rows.append({"pair": n, "prompt": prompt_text[prompt_id],
                           "better_domain_style": "", "better_prompt_match": ""})

    pd.DataFrame(key_rows).to_csv(OUT_DIR / "human_eval" / "answer_key.csv", index=False)
    for rater in RATERS:
        path = OUT_DIR / "human_eval" / "ratings" / f"ratings_{rater}.csv"
        if not path.exists():
            pd.DataFrame(sheet_rows).to_csv(path, index=False)
    print(f"25 blinded pairs written to {OUT_DIR / 'human_eval'}.")
    print("Each rater writes 'left' or 'right' in both columns of their own sheet, then rerun below.")

In [ ]:
key_path = OUT_DIR / "human_eval" / "answer_key.csv"
if not key_path.exists():
    print("No blinded comparison in this run — see the note above.")
else:
    key = pd.read_csv(key_path)
    rows = []
    for path in sorted((OUT_DIR / "human_eval" / "ratings").glob("ratings_*.csv")):
        sheet = pd.read_csv(path).fillna("").merge(key, on="pair")
        sheet = sheet[sheet["better_domain_style"].isin(["left", "right"]) &
                      sheet["better_prompt_match"].isin(["left", "right"])]
        if sheet.empty:
            continue

        def win_rate(column):
            return sheet.apply(lambda r: r[r[column] + "_model"] == comparison_run, axis=1).mean()

        rows.append({"rater": path.stem.replace("ratings_", ""),
                     "pairs_rated": len(sheet),
                     "run_wins_domain_style": round(win_rate("better_domain_style"), 3),
                     "run_wins_prompt_match": round(win_rate("better_prompt_match"), 3)})
    if rows:
        display(pd.DataFrame(rows))
    else:
        print("The sheets are still empty. Fill them in and rerun this cell.")

## Summary

### Publish the generated images and the model card (2.3.3, C3.2)

In [ ]:
image_dir = OUT_DIR / "hf_dataset"
shutil.rmtree(image_dir, ignore_errors=True)
image_dir.mkdir(parents=True)

with open(image_dir / "metadata.jsonl", "w") as handle:
    for row in generation_log.itertuples():
        name = f"{row.model}__{row.prompt_id}__seed{row.seed}.png"
        shutil.copy(row.path, image_dir / name)
        handle.write(json.dumps({"file_name": name, "model": row.model,
                                 "prompt_id": row.prompt_id, "prompt": row.prompt,
                                 "prompt_type": row.prompt_type, "seed": int(row.seed),
                                 "steps": int(row.steps), "guidance": float(row.guidance)}) + "\n")

api.create_repo(HF_IMAGE_REPO, repo_type="dataset", private=HF_PRIVATE, exist_ok=True)
api.upload_folder(folder_path=str(image_dir), repo_id=HF_IMAGE_REPO, repo_type="dataset")
print(f"pushed {len(generation_log)} images to {HF_IMAGE_REPO}")

final_row = results[results["model"] == FINAL_RUN].iloc[0]
base_row = results[results["model"] == "base"].iloc[0]
final_training = training_table[training_table["run"] == FINAL_RUN].iloc[0]

card = f"""---
license: cc-by-nc-sa-4.0
base_model: {BASE_MODEL}
tags: [stable-diffusion, lora, text-to-image, pokemon]
---

# {FINAL_RUN} — Pokemon-style LoRA for Stable Diffusion 1.5

LoRA adapter trained for DATA 266 Lab 2 (Pair_Programming_Team_33).

## Base model and data
Fine-tuned from `{BASE_MODEL}` at {RESOLUTION}x{RESOLUTION} on
[`{DATASET_NAME}`]({"https://huggingface.co/datasets/" + DATASET_NAME}): {len(train_records)}
captioned training images after removing {cleaning_summary["corrupted_removed"]} unusable and
{cleaning_summary["exact_duplicates_removed"] + cleaning_summary["perceptual_duplicates_removed"]}
duplicate images from {cleaning_summary["rows_raw"]} rows. The 90/10 split used seed {SEED}; the
test images were never trained on.

## Training settings
| setting | value |
|---|---|
| LoRA rank | {int(final_training["rank"])} |
| trainable parameters | {int(final_training["trainable_parameters"]):,} |
| learning rate | {final_training["learning_rate"]} |
| steps | {int(final_training["steps"])} |
| effective batch size | {int(final_training["effective_batch_size"])} |
| peak GPU memory | {final_training["peak_gpu_memory_gb"]} GB |
| training time | {final_training["gpu_hours"]} GPU-hours |

A second run at rank {[r for r in LORA_RANKS if f"lora_r{{r}}" != FINAL_RUN][0]} was trained on the
same data and the same step budget; this one was selected on in-domain KID.

## Results
| metric | base model | {FINAL_RUN} |
|---|---|---|
| KID, in-domain (lower is better) | {base_row["kid_in_domain"]} | {final_row["kid_in_domain"]} |
| CLIP score, all prompts | {base_row["clip_score"]} | {final_row["clip_score"]} |
| CLIP score, in-domain | {base_row["clip_in_domain"]} | {final_row["clip_in_domain"]} |
| LPIPS diversity between seeds | {base_row["lpips_diversity"]} | {final_row["lpips_diversity"]} |

## Limitations
- Trained on a small captioned set of one art style, so out-of-domain and compositional prompts
  drift back towards the base model or lose the style entirely.
- BLIP-generated captions are short and repetitive, which limits how precisely the adapter can be
  steered.
- KID over roughly {len(test_records)} test images is noisy; treat small differences as ties.

## License and intended use
The training data is `cc-by-nc-sa-4.0` and depicts copyrighted characters, so this adapter is for
non-commercial research only. Outputs were screened with the Stable Diffusion safety checker.
"""

(OUT_DIR / "runs" / "MODEL_CARD.md").write_text(card)
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md",
                repo_id=hf_model_repo(int(final_training["rank"])), repo_type="model")
print("model card pushed to", hf_model_repo(int(final_training["rank"])))

In [ ]:
print("Part 2 fine-tuning study — Shriram Dundigalla")
print("-" * 60)
print("dataset            :", DATASET_NAME)
print("cleaning           :", cleaning_summary["corrupted_removed"], "corrupted,",
      cleaning_summary["exact_duplicates_removed"] + cleaning_summary["perceptual_duplicates_removed"],
      "duplicates removed of", cleaning_summary["rows_raw"])
print("split (seed 266)   :", len(train_records), "train /", len(test_records), "test")
print("images generated   :", len(generation_log), "across", len(MODELS), "models")
print("final run          :", FINAL_RUN, "->", hf_model_repo(int(
    training_table.loc[training_table["run"] == FINAL_RUN, "rank"].iloc[0])))
print("image dataset      :", HF_IMAGE_REPO)
print("median s per image :", round(generation_log["seconds"].median(), 2))
print("safety-checker hits:", int(nsfw["flagged"].sum()), "of", len(nsfw))
print("outputs            :", OUT_DIR)
display(results)